# YgoMaster-PackBuilder — YGOPRODeck Conversion Tests

Notebook di test per il converter:

```text
YgoProDeckPackData
        ↓
convert_pack_cards_to_domain(...)
        ↓
list[domain.Card]
```

Obiettivi:

- verificare la conversione dei DTO YGOPRODeck in `Card`;
- preservare `name`, `ydk_id` e `source_rarity`;
- verificare che `rarity` resti `None`;
- verificare che `master_duel_id` resti `None`;
- controllare che l'ordine delle carte venga preservato;
- verificare che gli oggetti domain siano indipendenti dai DTO sorgente.


## 1. Ambiente e root repository

In [1]:
from pathlib import Path
import sys

print("Python interpreter:", sys.executable)
print("Current working directory:", Path.cwd())

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    repo_root = cwd
elif (cwd.parent / "pyproject.toml").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError(
        "Impossibile individuare la root della repository."
    )

print("Repository root:", repo_root)

Python interpreter: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\.venv\Scripts\python.exe
Current working directory: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\tests
Repository root: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder


## 2. Import

In [2]:
from copy import deepcopy

from ygomaster_packbuilder.domain import Card

from ygomaster_packbuilder.sources.ygoprodeck import (
    YgoProDeckPackCard,
    YgoProDeckPackData,
    convert_pack_cards_to_domain,
)

print("Imports OK")

Imports OK


## 3. Costruzione di un pack sorgente di test

In [3]:
source_pack = YgoProDeckPackData(
    external_id="24116",
    name="Scaled Invasion",
    description="Test pack",
    creator="livius03",
    image_id="8400623",
    cards=[
        YgoProDeckPackCard(
            name='"A" Cell Breeding Device',
            ydk_id=34541863,
            card_type="Continuous Spell",
            source_rarity="Common",
        ),
        YgoProDeckPackCard(
            name="Alien Ammonite",
            ydk_id=91123920,
            card_type="Tuner Monster",
            source_rarity="Rare",
        ),
    ],
)

print(source_pack)

YgoProDeckPackData(external_id='24116', name='Scaled Invasion', description='Test pack', creator='livius03', image_id='8400623', cards=[YgoProDeckPackCard(name='"A" Cell Breeding Device', ydk_id=34541863, card_type='Continuous Spell', source_rarity='Common'), YgoProDeckPackCard(name='Alien Ammonite', ydk_id=91123920, card_type='Tuner Monster', source_rarity='Rare')])


## 4. Conversione verso il domain

In [4]:
cards = convert_pack_cards_to_domain(source_pack)

assert isinstance(cards, list)
assert len(cards) == len(source_pack.cards)

print(cards)

[Card(name='"A" Cell Breeding Device', ydk_id=34541863, source_rarity='Common', master_duel_id=None, rarity=None), Card(name='Alien Ammonite', ydk_id=91123920, source_rarity='Rare', master_duel_id=None, rarity=None)]


## 5. Tipi risultanti

In [5]:
for card in cards:
    assert isinstance(card, Card)

print("All converted objects are domain.Card instances")

All converted objects are domain.Card instances


## 6. Preservazione dei dati della prima carta

In [6]:
source_card = source_pack.cards[0]
domain_card = cards[0]

assert domain_card.name == source_card.name
assert domain_card.ydk_id == source_card.ydk_id
assert domain_card.source_rarity == source_card.source_rarity

print(domain_card)

Card(name='"A" Cell Breeding Device', ydk_id=34541863, source_rarity='Common', master_duel_id=None, rarity=None)


## 7. Stato iniziale della carta domain

In [7]:
for card in cards:
    assert card.rarity is None
    assert card.master_duel_id is None

print("All cards are in Imported state")

All cards are in Imported state


## 8. Ordine delle carte preservato

In [8]:
source_names = [card.name for card in source_pack.cards]
domain_names = [card.name for card in cards]

assert domain_names == source_names

print(domain_names)

['"A" Cell Breeding Device', 'Alien Ammonite']


## 9. DTO e domain object sono istanze distinte

In [9]:
for source_card, domain_card in zip(source_pack.cards, cards):
    assert source_card is not domain_card

print("Source DTOs and domain cards are independent instances")

Source DTOs and domain cards are independent instances


## 10. Mutazioni successive del DTO non modificano il domain

In [10]:
test_pack = deepcopy(source_pack)

converted = convert_pack_cards_to_domain(test_pack)

original_name = converted[0].name
original_rarity = converted[0].source_rarity

test_pack.cards[0].name = "Modified Source Name"
test_pack.cards[0].source_rarity = "Ultra Rare"

assert converted[0].name == original_name
assert converted[0].source_rarity == original_rarity

print("Domain card remains unchanged after DTO mutation")

Domain card remains unchanged after DTO mutation


## 11. Pack senza carte

In [11]:
empty_pack = YgoProDeckPackData(
    external_id="empty",
    name="Empty Pack",
    description="",
    creator="tester",
    image_id="0",
    cards=[],
)

converted = convert_pack_cards_to_domain(empty_pack)

assert converted == []

print("Empty pack conversion OK")

Empty pack conversion OK


## 12. Source rarity preservata senza normalizzazione

In [12]:
test_pack = YgoProDeckPackData(
    external_id="rarity-test",
    name="Rarity Test",
    description="",
    creator="tester",
    image_id="0",
    cards=[
        YgoProDeckPackCard(
            name="Test Secret Rare",
            ydk_id=12345678,
            card_type="Effect Monster",
            source_rarity="Secret Rare",
        )
    ],
)

converted = convert_pack_cards_to_domain(test_pack)

assert converted[0].source_rarity == "Secret Rare"
assert converted[0].rarity is None

print(converted[0])

Card(name='Test Secret Rare', ydk_id=12345678, source_rarity='Secret Rare', master_duel_id=None, rarity=None)
